### Step 1: Load and verify the dataset

In [ ]:
from datasets import load_dataset

# Load the SQuAD v1.1 dataset using the specified Hugging Face repository ID
dataset = load_dataset("rajpurkar/squad", "plain_text")

# The dev set is typically the 'validation' split in the datasets library
dev_set = dataset["validation"]

# Show some samples from the dev set
print("\n--- Sample from SQuAD v1.1 Dev Set ---")
for i in range(3):
    print(f"\nSample {i+1}:")
    print(f"Context: {dev_set[i]['context'][:200]}...") # Truncate for brevity
    print(f"Question: {dev_set[i]['question']}")
    print(f"Answers: {dev_set[i]['answers']['text']}")

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]


--- Sample from SQuAD v1.1 Dev Set ---

Sample 1:
Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated...
Question: Which NFL team represented the AFC at Super Bowl 50?
Answers: ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']

Sample 2:
Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated...
Question: Which NFL team represented the NFC at Super Bowl 50?
Answers: ['Carolina Panthers', 'Carolina Panthers', 'Carolina Panthers']

Sample 3:
Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated...
Question: Where did Super

### Step 2: Create a reproducible sample
Start with 200 questions. Use a fixed seed so that your experiment can be reproduced.

In [ ]:
SEED = 42
SAMPLE_SIZE = 200

sample = (
    dev_set
    .shuffle(seed=SEED)
    .select(range(SAMPLE_SIZE))
)

print("Selected examples:", len(sample))
print(sample[0])

Selected examples: 200
{'id': '572759665951b619008f8884', 'title': 'Private_school', 'context': 'Private schooling in the United States has been debated by educators, lawmakers and parents, since the beginnings of compulsory education in Massachusetts in 1852. The Supreme Court precedent appears to favor educational choice, so long as states may set standards for educational accomplishment. Some of the most relevant Supreme Court case law on this is as follows: Runyon v. McCrary, 427 U.S. 160 (1976); Wisconsin v. Yoder, 406 U.S. 205 (1972); Pierce v. Society of Sisters, 268 U.S. 510 (1925); Meyer v. Nebraska, 262 U.S. 390 (1923).', 'question': 'In what year did Massachusetts first require children to be educated in schools?', 'answers': {'text': ['1852', '1852', '1852'], 'answer_start': [158, 158, 158]}}


However, SQuAD sometimes contains multiple questions for the same context. A stronger approach is to select questions with unique contexts:

In [ ]:
import random

SEED = 42
SAMPLE_SIZE = 200

random.seed(SEED)

indices = list(range(len(dev_set)))
random.shuffle(indices)

selected_examples = []
used_contexts = set()

for idx in indices:
    example = dev_set[idx]
    context = example["context"]

    if context not in used_contexts:
        selected_examples.append(example)
        used_contexts.add(context)

    if len(selected_examples) == SAMPLE_SIZE:
        break

print("Selected questions:", len(selected_examples))
print("Unique contexts:", len(used_contexts))

Selected questions: 200
Unique contexts: 200


### Step 4: Create random irrelevant contexts
The irrelevant context must:
- Come from another example
- Not be the original context
- Preferably have a different title
- Not contain any gold answer

In [ ]:
import random
import re

random.seed(SEED)


def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def contains_gold_answer(context, answers):
    normalized_context = normalize_text(context)

    for answer in answers:
        normalized_answer = normalize_text(answer)

        if not normalized_answer:
            continue

        pattern = r"\b" + re.escape(normalized_answer) + r"\b"

        if re.search(pattern, normalized_context):
            return True

    return False


def find_random_irrelevant_context(example, context_pool):
    candidates = [
        candidate
        for candidate in context_pool
        if candidate["context"] != example["context"]
        and candidate["title"] != example["title"]
        and not contains_gold_answer(
            candidate["context"],
            example["answers"]["text"]
        )
    ]

    return random.choice(candidates)


random_conditions = []

for example in selected_examples:
    distractor = find_random_irrelevant_context(
        example,
        selected_examples
    )

    random_conditions.append({
        "id": example["id"],
        "question": example["question"],
        "answers": example["answers"]["text"],
        "relevant_context": example["context"],
        "irrelevant_context": distractor["context"],
        "original_title": example["title"],
        "distractor_title": distractor["title"]
    })

print(random_conditions[0])

{'id': '57296a65af94a219006aa3c5', 'question': 'How does the secondary theory say most cpDNA is structured?', 'answers': ['linear', 'linear', 'linear'], 'relevant_context': 'In cpDNA, there are several A → G deamination gradients. DNA becomes susceptible to deamination events when it is single stranded. When replication forks form, the strand not being copied is single stranded, and thus at risk for A → G deamination. Therefore, gradients in deamination indicate that replication forks were most likely present and the direction that they initially opened (the highest gradient is most likely nearest the start site because it was single stranded for the longest amount of time). This mechanism is still the leading theory today; however, a second theory suggests that most cpDNA is actually linear and replicates through homologous recombination. It further contends that only a minority of the genetic material is kept in circular chromosomes while the rest is in branched, linear, or other com

### Important limitation
Because your sample has only 200 passages, the possible distractor pool is relatively small. It is better to construct distractors using all unique contexts in the validation set:

In [ ]:
context_pool = []
seen_contexts = set()

for item in dev_set:
    context = item["context"]

    if context not in seen_contexts:
        context_pool.append({
            "title": item["title"],
            "context": context
        })
        seen_contexts.add(context)

print("Unique contexts:", len(context_pool))

Unique contexts: 2067


In [ ]:
distractor = find_random_irrelevant_context(
    example,
    context_pool
)

### Step 5: Create related irrelevant contexts

A random unrelated passage may be too easy for the model to ignore. The more interesting condition is a passage that looks relevant but does not contain the answer.
You can retrieve related passages using TF-IDF:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re


def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def contains_gold_answer(context, answers):
    normalized_context = normalize_text(context)

    return any(
        normalize_text(answer) in normalized_context
        for answer in answers
        if normalize_text(answer)
    )


corpus = [
    f"{item['title']} {item['context']}"
    for item in context_pool
]

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=30000,
    ngram_range=(1, 2)
)

context_vectors = vectorizer.fit_transform(corpus)

In [ ]:
def find_related_distractor(example, top_k=100):
    query_text = (
        f"{example['title']} "
        f"{example['question']} "
        f"{example['context'][:300]}"
    )

    query_vector = vectorizer.transform([query_text])

    similarities = cosine_similarity(
        query_vector,
        context_vectors
    ).flatten()

    ranked_indices = np.argsort(similarities)[::-1]

    # Priority 1: another paragraph from the same article
    for idx in ranked_indices:
        candidate = context_pool[idx]

        if candidate["context"] == example["context"]:
            continue

        if candidate["title"] != example["title"]:
            continue

        if contains_gold_answer(
            candidate["context"],
            example["answers"]["text"]
        ):
            continue

        return {
            "context": candidate["context"],
            "title": candidate["title"],
            "similarity": float(similarities[idx]),
            "source": "same_title"
        }

    # Priority 2: related paragraph from another article
    for idx in ranked_indices[:top_k]:
        candidate = context_pool[idx]

        if candidate["context"] == example["context"]:
            continue

        if contains_gold_answer(
            candidate["context"],
            example["answers"]["text"]
        ):
            continue

        return {
            "context": candidate["context"],
            "title": candidate["title"],
            "similarity": float(similarities[idx]),
            "source": "tfidf_fallback"
        }

    return None

In [ ]:
# Apply it to the 200 questions:

experimental_data = []

for example in selected_examples:
    random_distractor = find_random_irrelevant_context(
        example,
        context_pool
    )

    related_distractor = find_related_distractor(example)

    experimental_data.append({
        "id": example["id"],
        "title": example["title"],
        "question": example["question"],
        "gold_answers": example["answers"]["text"],
        "relevant_context": example["context"],

        "random_irrelevant_context":
            random_distractor["context"],

        "random_irrelevant_title":
            random_distractor["title"],

        "related_irrelevant_context": (
            related_distractor["context"]
            if related_distractor else None
        ),

        "related_irrelevant_title": (
            related_distractor["title"]
            if related_distractor else None
        ),

        "related_similarity": (
            related_distractor["similarity"]
            if related_distractor else None
        ),

        "related_source": (
            related_distractor["source"]
            if related_distractor else None
        )
    })

In [ ]:
leakage_cases = []

for example in experimental_data:
    if contains_gold_answer(
        example["related_irrelevant_context"],
        example["gold_answers"]
    ):
        leakage_cases.append(example["id"])

print("Answer-leakage cases:", len(leakage_cases))
print(leakage_cases[:10])

Answer-leakage cases: 0
[]


In [ ]:
random_leakage_cases = []

for example in experimental_data:
    if contains_gold_answer(
        example["random_irrelevant_context"],
        example["gold_answers"]
    ):
        random_leakage_cases.append(example["id"])

print("Random-context leakage cases:", len(random_leakage_cases))

Random-context leakage cases: 0


In [ ]:
# Check how many used same-title distractors

from collections import Counter

source_counts = Counter(
    item["related_source"]
    for item in experimental_data
)

print(source_counts)

Counter({'same_title': 200})


In [ ]:
import random

random.seed(42)
validation_sample = random.sample(experimental_data, 30)

for number, example in enumerate(validation_sample, start=1):
    print("\n" + "=" * 100)
    print(f"EXAMPLE {number}")

    print("ORIGINAL TITLE:")
    print(example["title"])

    print("\nQUESTION:")
    print(example["question"])

    print("\nGOLD ANSWERS:")
    print(example["gold_answers"])

    print("\nRELATED DISTRACTOR TITLE:")
    print(example["related_irrelevant_title"])

    print("\nRETRIEVAL SOURCE:")
    print(example["related_source"])

    print("\nSIMILARITY:")
    print(example["related_similarity"])

    print("\nRELATED DISTRACTOR:")
    print(example["related_irrelevant_context"][:700])


EXAMPLE 1
ORIGINAL TITLE:
Civil_disobedience

QUESTION:
If the head of government refuses to enforce a decision of the highest court what terminology could be used?

GOLD ANSWERS:
['constitutional impasse', 'her or his capacity as public official', 'capacity as public official', 'constitutional impasse']

RELATED DISTRACTOR TITLE:
Civil_disobedience

RETRIEVAL SOURCE:
same_title

SIMILARITY:
0.1982929553654659

RELATED DISTRACTOR:
Non-revolutionary civil disobedience is a simple disobedience of laws on the grounds that they are judged "wrong" by an individual conscience, or as part of an effort to render certain laws ineffective, to cause their repeal, or to exert pressure to get one's political wishes on some other issue. Revolutionary civil disobedience is more of an active attempt to overthrow a government (or to change cultural traditions, social customs, religious beliefs, etc...revolution doesn't have to be political, i.e. "cultural revolution", it simply implies sweeping and wi

In [ ]:
import re


def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def contains_gold_answer(context, answers):
    normalized_context = normalize_text(context)

    for answer in answers:
        normalized_answer = normalize_text(answer)

        if not normalized_answer:
            continue

        pattern = r"\b" + re.escape(normalized_answer) + r"\b"

        if re.search(pattern, normalized_context):
            return True

    return False

### Step 6: Manually validate the distractors
Before calling an LLM, inspect at least 20–30 examples:

In [ ]:
for example in experimental_data[:10]:
    print("\n" + "=" * 80)
    print("QUESTION:")
    print(example["question"])

    print("\nGOLD ANSWERS:")
    print(example["gold_answers"])

    print("\nRELATED IRRELEVANT CONTEXT:")
    print(example["related_irrelevant_context"][:500])


QUESTION:
How does the secondary theory say most cpDNA is structured?

GOLD ANSWERS:
['linear', 'linear', 'linear']

RELATED IRRELEVANT CONTEXT:
In Germany, teachers are mainly civil servants recruited in special university classes, called Lehramtstudien (Teaching Education Studies). There are many differences between the teachers for elementary schools (Grundschule), lower secondary schools (Hauptschule), middle level secondary schools (Realschule) and higher level secondary schools (Gymnasium). Salaries for teachers depend on the civil servants' salary index scale (Bundesbesoldungsordnung).

QUESTION:
What is the name of the two-tentacled cydippid that feedsentirely on salps called?

GOLD ANSWERS:
['Lampea', 'Lampea', 'Lampea']

RELATED IRRELEVANT CONTEXT:
In Tibetan Buddhism the teachers of Dharma in Tibet are most commonly called a Lama. A Lama who has through phowa and siddhi consciously determined to be reborn, often many times, in order to continue their Bodhisattva vow is call

In [ ]:
pilot_data = experimental_data[:20]

In [ ]:
import random

random.seed(42)

validation_sample = random.sample(experimental_data, 30)

for number, example in enumerate(validation_sample, start=1):
    print("\n" + "=" * 100)
    print(f"EXAMPLE {number}")
    print("\nQUESTION:")
    print(example["question"])

    print("\nGOLD ANSWERS:")
    print(example["gold_answers"])

    print("\nRELEVANT CONTEXT:")
    print(example["relevant_context"][:500])

    print("\nRANDOM IRRELEVANT CONTEXT:")
    print(example["random_irrelevant_context"][:500])

    print("\nRELATED IRRELEVANT CONTEXT:")
    print(example["related_irrelevant_context"][:500])

    print("\nSIMILARITY:")
    print(example["related_similarity"])


EXAMPLE 1

QUESTION:
If the head of government refuses to enforce a decision of the highest court what terminology could be used?

GOLD ANSWERS:
['constitutional impasse', 'her or his capacity as public official', 'capacity as public official', 'constitutional impasse']

RELEVANT CONTEXT:
Civil disobedience is usually defined as pertaining to a citizen's relation to the state and its laws, as distinguished from a constitutional impasse in which two public agencies, especially two equally sovereign branches of government, conflict. For instance, if the head of government of a country were to refuse to enforce a decision of that country's highest court, it would not be civil disobedience, since the head of government would be acting in her or his capacity as public official rather 

RANDOM IRRELEVANT CONTEXT:
On the other hand, in the late 1980s the Western Atlantic ctenophore Mnemiopsis leidyi was accidentally introduced into the Black Sea and Sea of Azov via the ballast tanks of ships

In [ ]:
import json

with open(
    "squad_200_experimental_data.jsonl",
    "w",
    encoding="utf-8"
) as file:
    for example in experimental_data:
        file.write(
            json.dumps(example, ensure_ascii=False) + "\n"
        )

print("Saved examples:", len(experimental_data))

Saved examples: 200


In [ ]:
import random
import json

random.seed(42)

validation_sample = random.sample(
    experimental_data,
    30
)

validation_records = []

for number, example in enumerate(validation_sample, start=1):
    validation_records.append({
        "example_number": number,
        "id": example["id"],
        "title": example["title"],
        "question": example["question"],
        "gold_answers": example["gold_answers"],
        "related_distractor":
            example["related_irrelevant_context"],
        "similarity":
            example["related_similarity"],
        "manual_label": "",
        "manual_reason": ""
    })

with open(
    "manual_validation_30.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        validation_records,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Saved validation examples:", len(validation_records))

Saved validation examples: 30
